In [0]:
%pip install openai unidecode gspread==5.12.4 tiktoken mlflow
%restart_python
%load_ext autoreload
%autoreload 2 

In [0]:
from abc import ABC, abstractmethod
import time
import mlflow
from contextlib import contextmanager
import json
import pandas as pd
import datetime
import re
import openai
from openai import OpenAI
import gspread
import random
import logging
import os
from unidecode import unidecode
from pathlib import Path
import pyspark
from pyspark.sql.functions import *
from functools import reduce
from typing import *
import tiktoken
import sys

In [0]:

# Add the parent directory of 'localizers' to sys.path
project_root = Path('/Workspace/Users/krista@jamcity.com/CentralizedLocalizationWorkflow/localizers')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

### Localizer Modules 
from general_config import *
from ml_tracker import *
from generic_localizer import *
from in_game_config import *
from in_game_localizer import *

In [0]:
%run "./authenticationScript"

In [0]:
#TODO update authenticatinon script and secrets!!
gsheet_client = get_gspread_client_from_secret_old()
gpt_client = get_model_client() 

In [0]:
sh = gsheet_client.open_by_url("https://docs.google.com/spreadsheets/d/18O9bAuufPTOawCI6I-Of9xlqafqXiz-ino0iGgr6Kb0/edit?gid=1917862618#gid=1917862618")

In [0]:
wksht = sh.worksheet("input")

In [0]:
input_data = wksht.get_all_values()

cols, data = input_data[0], input_data[1:]
df = spark.createDataFrame(data, schema=cols)

In [0]:
item_desc = df.where("KEY like '%ITEM_DESC_%'").toPandas()
other = df.where("KEY not like '%ITEM_DESC_%'").toPandas() #.display()

In [0]:
#### 

def build_context_enrichment_prompt():

  # peform preprocessing
  PREPROCESSING_PROMPT = f""" You are helping preprocess item-description data for a mobile game called Disney Magic Match.

  Each input row contains:
  - KEY: an internal item key
  - en_US: an English flavor text line

  The English flavor text may include puns, wordplay, humor, or figurative language. Your task is NOT to explain the joke. Your task is to identify the underlying object being described and generate structured preprocessing fields.

  For each row, produce the following new columns:

  1. object
  - Identify the core object being described.
  - Infer it primarily from the KEY, using the English phrase only as supporting context.
  - Use a short singular noun phrase.
  - Normalize the object into a clean, simple label.
  - Examples:
    - "dumbbell"
    - "wide-brim hat"
    - "bowler hat"
    - "surfboard"
    - "music note"
    - "treasure chest"

  2. descriptor
  - Identify the descriptor attached to the object, if present.
  - Usually this comes from the KEY and may refer to color, material, style, shape, tone, size, or other modifier.
  - Examples:
    - "blue"
    - "purple"
    - "silver"
    - "light gold"
    - "round"
    - "wide-brim"
    - "murky"
  - If no useful descriptor is present, output an empty string.

  3. additional_context
  - Add short helpful inferred context ABOUT the object itself.
  - This should describe what the object is, what it is used for, or what kind of thing it is.
  - Do NOT explain the joke or pun.
  - Do NOT repeat the object or descriptor unless necessary.
  - This field is optional. If there is no useful extra context, output an empty string.
  - Good example:
    - For "wide-brim hat": "A style of hat with a large brim, often used for shade or sun protection."
  - Other examples:
    - "A handheld controller used to play video games."
    - "A tool with two blades used for cutting paper or other materials."
    - "A musical symbol used to represent pitch or rhythm."

  4. simple_description_en
  - Write a plain, literal English description of the item using the object, descriptor, and any relevant context.
  - This should be simple and clear, not funny.
  - Do NOT preserve the pun or wordplay.
  - Keep it concise, usually one sentence or sentence fragment.
  - Examples:
    - "A blue dumbbell used for strength training."
    - "A purple wide-brim hat that provides shade from the sun."
    - "A silver drinking glass for serving beverages."

  General rules:
  - Prioritize accuracy and consistency over creativity.
  - Base the answer mainly on the KEY, then refine with the English flavor text if helpful.
  - Ignore the humorous wording when identifying the object.
  - Normalize object names so similar items use the same naming style across rows.
  - Use sentence case for descriptions.
  - Do not invent lore, story context, or visual details that are not reasonably inferable.
  - If the KEY implies a stylized or unusual color for a real-world object, keep that descriptor.
  - If a gemstone or item name is specific (for example, "emerald"), preserve that specificity in the object when useful.
  - If the descriptor is actually a material or style rather than a color, record it as written in natural English.
  - If the object name in the KEY is camelCase, convert it into normal readable English.
    - Example: "hatWideBrim" -> "wide-brim hat"
    - Example: "tableRound" -> "round table"
    - Example: "musicNote2" -> "music note"

  Do not include any extra commentary.
  Do not include markdown fences.
  Do not include the original input fields unless explicitly requested. 


  Output format:
  """

  PREPROCESSING_PROMPT += f"""
              Respond in **JSON format**, one object per row:
              json
              [
              {{ "key":"key 1",
                  "object": "object 1",
                  "descriptor": "descriptor 1",
                  "additional_context":  "additional context 1",
                  "simple_description_en":"simple description 1" }},
              {{ "key":"key 2",
                  "object": "object 2",
                  "descriptor": "descriptor 2",
                  "additional_context":  "additional context 2",
                  "simple_description_en":"simple description 2" }},,
              ...
              ]\n\n
              """
  return PREPROCESSING_PROMPT


def prepare_data_json_for_enrichment(item_desc):
    PH_RE = re.compile(r"<[^>]+>|\{[^}]+\}")
    payload = []
    for _, r in item_desc.iterrows():
        en = r.get("en_US", "") or ""
        key = r.get("KEY")
        item = {
            "en_US": en,
            'key':key
            # keep payload tiny; include only relevant hints
        }
        # include char_limit only if it’s a finite number
        #limit = r.get("char_limit","") or ""
        #if pd.notna(limit):
        #    item["char_limit"] = int(r['char_limit'])
        
        ## Grab the other cols and convert to string
        #other_cols.remove('char_limit')
        #for col in other_cols:
        #    val = r.get(col)
        #    if pd.notna(val):
        #        item[col] = str(val)
            
            
        # placeholders list (helps the model self-check)
        ph = PH_RE.findall(en)
        if ph:
            item["placeholders"] = ph
        # tiny checksum for alignment debugging
        #if "src_hash8" in df.columns and r.get("src_hash8"):
        #    item["src_hash8"] = r["src_hash8"]

        payload.append(item)


    prepped = json.dumps(payload)
    return prepped



def preprocess_item_desc(df):


    PREPROCESSING_PROMPT = build_context_enrichment_prompt()
    data_json = prepare_data_json_for_enrichment(df)
    return [
            {"role": "system", "content": PREPROCESSING_PROMPT},
            {"role": "user",   "content":data_json}
    ]




In [0]:
prompt = preprocess_item_desc(item_desc)

In [0]:
def run_context_enrichment(prompt):

        MODEL = "gpt-5.4"
        temperature = 0.05

        response = gpt_client.chat.completions.create(
                model=MODEL, 
                messages=prompt,
                temperature=temperature  # adjust for creativity vs. stability
        )

        output = response.choices[0].message.content
        usage = response.usage
        
        return output, usage

In [0]:
def _parse_model_json_block(raw_output: str):
        """
        Clean and parse a JSON-like string from a model output wrapped in markdown code block.
        """
        try:
            # Strip markdown-style code block markers like ```json ... ```
            cleaned = re.sub(r"^```json|```$", "", raw_output.strip(), flags=re.IGNORECASE).strip()
            # Remove escaped newlines
            # PA 2025-11-06: TEMPORARY FIX, NEEDS MORE ANALYSIS - example: "<size=150%> %v0% \n<size=100%>off!". -->
            #cleaned = cleaned.replace("\\n", "").replace("\n", "").strip()
            # PA 2025-11-06: TEMPORARY FIX, NEEDS MORE ANALYSIS - example: "<size=150%> %v0% \n<size=100%>off!". <--
            loaded = json.loads(cleaned)
        except Exception as e:
            raise ValueError(f"Could not parse JSON: {e}")

        if isinstance(loaded, str):
            try:
                return json.loads(loaded)
            except Exception as e:
                raise ValueError(f"Could not parse inner JSON: {e}")
        return loaded
    

In [0]:
parsed = _parse_model_json_block(output)

parsed_df = pd.DataFrame(parsed)

In [0]:
def write_preprocessed(sh, parsed_df):
    num_cols= len(list(parsed_df.columns))
    num_rows = len(parsed_df.values.tolist())+1
    try:
        sh.add_worksheet('preprocessed', rows=num_rows, cols=num_cols)
    except Exception as e:
        pass
    preprocessed = sh.worksheet('preprocessed')

    headers = parsed_df.columns.tolist()
    out_data = parsed_df.values.tolist()


    number_to_letter = {
        "1": "A",  "2": "B",  "3": "C",  "4": "D",  "5": "E",
        "6": "F",  "7": "G",  "8": "H",  "9": "I", "10": "J",
        "11": "K", "12": "L", "13": "M", "14": "N", "15": "O",
        "16": "P", "17": "Q", "18": "R", "19": "S", "20": "T",
        "21": "U", "22": "V", "23": "W", "24": "X", "25": "Y",
        "26": "Z", "27": "AA", "28": "AB", "29": "AC", "30": "AD",
        "31": "AE", "32": "AF", "33": "AG", "34": "AH", "35": "AI",
        "36": "AJ", "37": "AK", "38": "AL", "39": "AM", "40": "AN",
        "41": "AO", "42": "AP", "43": "AQ", "44": "AR", "45": "AS",
        "46": "AT", "47": "AU", "48": "AV", "49": "AW", "50": "AX",
        "51": "AY", "52": "AZ", "53": "BA", "54": "BB", "55": "BC",
        "56": "BD", "57": "BE", "58": "BF", "59": "BG", "60": "BH",
    }
    letter_range = number_to_letter[str(len(headers))]
    data_range = f"A2:{letter_range}{len(out_data)+1}"


    preprocessed.batch_update([{
                    'range':f"A1:{letter_range}1", 'values':[headers]
                    },
                    {
                    'range':data_range, 'values':out_data
                    }])
    
    return "Done writing preprocessed!"

In [0]:
num_cols= len(list(parsed_df.columns))
num_rows = len(parsed_df.values.tolist())+1
try:
    sh.add_worksheet('preprocessed', rows=num_rows, cols=num_cols)
except Exception as e:
    pass
preprocessed = sh.worksheet('preprocessed')#.update_rows(parsed,



In [0]:
preprocessed#.update_rows(parsed,

In [0]:
headers = parsed_df.columns.tolist()
out_data = parsed_df.values.tolist()


number_to_letter = {
    "1": "A",  "2": "B",  "3": "C",  "4": "D",  "5": "E",
    "6": "F",  "7": "G",  "8": "H",  "9": "I", "10": "J",
    "11": "K", "12": "L", "13": "M", "14": "N", "15": "O",
    "16": "P", "17": "Q", "18": "R", "19": "S", "20": "T",
    "21": "U", "22": "V", "23": "W", "24": "X", "25": "Y",
    "26": "Z", "27": "AA", "28": "AB", "29": "AC", "30": "AD",
    "31": "AE", "32": "AF", "33": "AG", "34": "AH", "35": "AI",
    "36": "AJ", "37": "AK", "38": "AL", "39": "AM", "40": "AN",
    "41": "AO", "42": "AP", "43": "AQ", "44": "AR", "45": "AS",
    "46": "AT", "47": "AU", "48": "AV", "49": "AW", "50": "AX",
    "51": "AY", "52": "AZ", "53": "BA", "54": "BB", "55": "BC",
    "56": "BD", "57": "BE", "58": "BF", "59": "BG", "60": "BH",
}
letter_range = number_to_letter[str(len(headers))]
data_range = f"A2:{letter_range}{len(out_data)+1}"


preprocessed.batch_update([{
                'range':f"A1:{letter_range}1", 'values':[headers]
                },
                {
                'range':data_range, 'values':out_data
                }])


In [0]:
def translate_to_fun_description(preprocessed_df, language_cd):
    
    #language_prompt = HUMOR_LANGUAGE_PROMPT[language_cd]

    return

def translate_to_object_based_quip(preprocessed_df, language_cd)

    #language_prompt = HUMOR_LANGUAGE_PROMPT[language_cd]

    return